## 00 — Designing the LOD Pipeline

Before we write a single output file, we design the pipeline. Good engineering starts with the decision, not the code.

We need to answer four questions:

1. What epsilon value should each LOD level use?
2. Which zoom range should each level serve?
3. What do we do with features that simplify down to fewer than 2 points?
4. Should we filter by feature importance at coarse levels, or just simplify geometry?

## The Four Levels

Standard web maps use zoom levels 0–18. The full railroad dataset is only useful at zoom 7 and above — at lower zooms, the detail is invisible.

We divide the zoom range into four bands:

| Level | Zoom | What the user sees | Data priority |
|-------|------|---------------------|---------------|
| Coarse | 1–3 | Entire world | Major trunk lines only |
| Medium | 4–6 | Continents / regions | Most named lines |
| Fine | 7–10 | Countries / states | Full network |
| Extra Fine | 11+ | Cities / local | Full detail |

Each level needs an epsilon that makes sense for its zoom range.

## Choosing Epsilon

Epsilon is in **degrees** because our coordinates are longitude/latitude.

One degree of longitude at the equator is approximately 111 km. At higher latitudes it is less. For our purposes, a rough guide:

| Epsilon | Roughly... | Suitable for |
|---------|------------|--------------|
| 1.0 | ~111 km | World view (zoom 1–3) |
| 0.1 | ~11 km | Continental (zoom 4–6) |
| 0.01 | ~1 km | Regional / country (zoom 7–10) |
| 0.001 | ~100 m | Local / city (zoom 11+) |

These are starting points. The right values depend on visual judgment — which is why we compared them on the map in the previous module.

## Edge Case — Features That Collapse

At high epsilon values, some short or nearly-straight features will simplify down to fewer than 2 coordinate pairs. A `LineString` with 0 or 1 points is not valid GeoJSON.

We need a rule for this. There are two options:

**Option A — Drop collapsed features entirely.**
They are too small to matter at this zoom level anyway. This keeps output files clean.

**Option B — Keep them as 2-point lines.**
A feature that simplifies to a single point was already nearly a straight line. We could force-keep start and end even if they are identical. But that produces degenerate geometry.

We will use **Option A**. Any feature whose simplified form has fewer than 2 points is dropped from that LOD level.

In [1]:
import json
from pathlib import Path
from shapely.geometry import LineString

data_path = Path("../../data/ne_10m_railroads.geojson")

with open(data_path) as f:
    railroads = json.load(f)

features = railroads["features"]
print(f"Loaded {len(features):,} features")

Loaded 25,413 features


In [2]:
# Check how many features collapse at each epsilon
epsilons = {"coarse": 1.0, "medium": 0.1, "fine": 0.01, "extra_fine": 0.001}

print(f"{'Level':<12} {'Epsilon':>8} {'Kept':>8} {'Dropped':>9} {'Drop %':>8}")
print("-" * 52)

for name, eps in epsilons.items():
    kept = 0
    dropped = 0
    for f in features:
        coords = f["geometry"]["coordinates"]
        simplified = LineString(coords).simplify(eps, preserve_topology=False)
        if len(simplified.coords) >= 2:
            kept += 1
        else:
            dropped += 1
    pct = dropped / len(features) * 100
    print(f"{name:<12} {eps:>8} {kept:>8,} {dropped:>9,} {pct:>7.1f}%")

Level         Epsilon     Kept   Dropped   Drop %
----------------------------------------------------
coarse            1.0   25,413         0     0.0%
medium            0.1   25,413         0     0.0%
fine             0.01   25,413         0     0.0%
extra_fine      0.001   25,413         0     0.0%


## Edge Case — Feature Importance at Coarse Levels

Even after simplification, the coarse level may include thousands of minor local railroad spurs. At zoom 1, a spur in suburban Tokyo is noise — it adds rendering cost without adding useful information.

The `scalerank` property is Natural Earth's built-in importance ranking. Lower values are more important:

| scalerank | Intended display scale |
|-----------|------------------------|
| 1–3 | Global / continental |
| 4–6 | Country / regional |
| 7–10 | Local |

For the coarse level, we can filter to only `scalerank <= 4`. Let's see how many features that leaves.

In [3]:
for max_rank in [3, 4, 5, 6, 10]:
    count = sum(1 for f in features if f["properties"]["scalerank"] <= max_rank)
    pct = count / len(features) * 100
    print(f"scalerank <= {max_rank:2d}: {count:>6,} features ({pct:.1f}%)")

scalerank <=  3:      0 features (0.0%)
scalerank <=  4:  2,845 features (11.2%)
scalerank <=  5:  3,443 features (13.5%)
scalerank <=  6:  5,946 features (23.4%)
scalerank <= 10: 25,413 features (100.0%)


## The Pipeline Design

Based on the numbers above, here is the plan we will implement:

```
for each LOD level:
    1. filter features by scalerank (coarse only)
    2. simplify each feature's geometry with Shapely at the level's epsilon
    3. discard any feature whose simplified geometry has < 2 points
    4. write the surviving features to a new GeoJSON file
```

**Why Shapely instead of our own implementation?**

We built our own D-P to understand it. We verified it matches Shapely. For a pipeline that processes 25,000 features, Shapely is 10–20x faster because it calls compiled C code. Using the faster tool for bulk work is not laziness — it is engineering judgment.

## Exercise A

Look at the collapse numbers from the table above. At `epsilon=1.0`, what fraction of the original 25,000+ features survive?

Now reconsider: should the coarse LOD also apply a `scalerank` filter, or is geometry simplification alone enough to reduce data volume to a usable level? Write your argument in a comment.

In [4]:
# Write your argument as comments
# Consider: what is the goal of the coarse level — correctness or speed?
# Your code here
survived = 25413
total = 25413
fraction = survived / total * 100
print(f"At epsilon=1.0: {survived:,} of {total:,} features survive ({fraction:.1f}%)")

# so 100% survive. shapely never kills a line below 2 points, it just
# squashes them down. cool but kinda useless for the coarse level.
#
# 25k features is still a lot for zoom 1-3 imo. like the whole world fits
# on screen, you cant even see most of these tiny spurs. browser still has
# to draw them tho which is wasted work.
#
# i think we should add the scalerank filter on top. scalerank <= 4 drops
# it from 25k to 2845, thats like 9x less stuff. and the ones we drop are
# the small ones nobody can see at world zoom anyway so it doesnt look
# worse.
#
# tldr: simplify shrinks each line, scalerank cuts the count. coarse level
# needs both or its still slow.

At epsilon=1.0: 25,413 of 25,413 features survive (100.0%)


## Exercise B

The `natlscale` property stores the intended display scale for each feature (e.g. `250` = designed for 1:250,000 maps).

Could `natlscale` be used as a LOD filter instead of `scalerank`? Write code that shows the distribution of `natlscale` values, then make a case for or against using it.

In [5]:
# Show distribution of natlscale values
# Your code here
from collections import Counter

natlscales = [f["properties"]["natlscale"] for f in features]
counter = Counter(natlscales)

print(f"{'natlscale':>12} {'count':>8}")
print("-" * 25)
for val in sorted(counter.keys()):
    print(f"{val:>12} {counter[val]:>8}")

print(f"\ntotal: {len(natlscales)} features, {len(counter)} unique values")

# could natlscale work as a filter instead of scalerank? kinda yeah but
# 
#
# pros: its a continuous-ish number so you could pick any threshold you
# want. more flexible than scalerank's 1-10 buckets.
#
# cons: more values means more guessing. with scalerank i can just go
# "<=4 for coarse" and its done. with natlscale i gotta look at the
# distribution every time and figure out what number means "important
# enough". also natlscale is about map scale (1:250000 etc) not really
# about importance, so its a different vibe.
#
# id stick with scalerank for filtering. natlscale is more useful as
# extra info maybe but not the main filter.

   natlscale    count
-------------------------
           1     7759
           5     4721
          10     3871
          20     3116
          40     2503
          75      598
         150     2845

total: 25413 features, 7 unique values


## Check Your Understanding

We are dropping features that collapse to fewer than 2 points after simplification.

A classmate argues: *"We should never drop features — even a tiny spur might be important to someone."*

Write a 2–3 sentence response that explains why dropping collapsed features is the correct engineering decision for a zoom-level-based LOD system.

---

The classmate has a point but theyre missing how zoom levels work. A feature that collapses to fewer than 2 points at coarse epsilon is already smaller than a pixel at that zoom — nobody can see it anyway, so dropping it doesnt actually hide anything from the user. Plus that same feature is still in the medium/fine/extra fine LOD files at the right zoom, so if someone zooms in to look for it, its right there. Were not deleting data, were just not loading the invisible stuff at world view.

## Next

In [01 — Writing the LOD Files](./01-Writing_the_LOD_Files.ipynb), we implement the pipeline and write the four output files to disk.